In [ ]:
import os, json
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/PACL_Project"
PHASE3_DIR = f"{PROJECT_DIR}/phase3_outputs"
PHASE4_DIR = f"{PROJECT_DIR}/phase4_outputs"
CKPT_DIR = f"{PHASE4_DIR}/checkpoints_mlp_fusion"
os.makedirs(PHASE4_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

train_records = load_jsonl(f"{PHASE3_DIR}/train.jsonl")
test_closed_records = load_jsonl(f"{PHASE3_DIR}/test_closed.jsonl")
test_open_records = load_jsonl(f"{PHASE3_DIR}/test_open.jsonl")

poet_to_id = {p: i for i, p in enumerate(sorted(set(r["poet"] for r in train_records)))}
n_train_poets = len(poet_to_id)
print(f"train: {len(train_records)} نمونه / {n_train_poets} شاعر")
assert n_train_poets == 150, (
    f"❌ انتظار داشتیم دقیقاً ۱۵۰ شاعر در train، شد {n_train_poets}."
)
print(f"test_closed: {len(test_closed_records)} نمونه")
print(f"test_open:   {len(test_open_records)} نمونه")

train_texts = [r["text"] for r in train_records]
train_poets = [r["poet"] for r in train_records]
train_poem_ids = [r["poem_id"] for r in train_records]
train_labels = [poet_to_id[p] for p in train_poets]


In [ ]:
import numpy as np

train_embeddings = np.load(f"{PHASE3_DIR}/train_embeddings_byt5base.npy")
test_closed_embeddings = np.load(f"{PHASE3_DIR}/test_closed_embeddings_byt5base.npy")
test_open_embeddings = np.load(f"{PHASE3_DIR}/test_open_embeddings_byt5base.npy")

print("شکلِ embeddingِ train:", train_embeddings.shape)
print("شکلِ embeddingِ test_closed:", test_closed_embeddings.shape)
print("شکلِ embeddingِ test_open:", test_open_embeddings.shape)

assert train_embeddings.shape[0] == len(train_records)
assert test_closed_embeddings.shape[0] == len(test_closed_records)
assert test_open_embeddings.shape[0] == len(test_open_records)
assert np.isnan(train_embeddings).any() == False
assert np.isnan(test_closed_embeddings).any() == False
assert np.isnan(test_open_embeddings).any() == False
print("✅ هیچ NaN در هیچ‌کدام از سه embedding وجود ندارد.")


In [ ]:
import time
from sklearn.feature_extraction.text import TfidfVectorizer

t0 = time.time()
vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), max_features=50000, sublinear_tf=True, dtype=np.float32)
vectorizer.fit(train_texts)
print(f"TF-IDF رویِ Train فیت شد. اندازه‌یِ واژگان: {len(vectorizer.vocabulary_)}  ({time.time()-t0:.1f}s)")

train_tfidf = vectorizer.transform(train_texts)
print("شکلِ TF-IDFِ train:", train_tfidf.shape, f"  ({time.time()-t0:.1f}s)")


In [ ]:
import time
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize as sk_normalize

t0 = time.time()
SVD_DIM = 300
svd = TruncatedSVD(n_components=SVD_DIM, random_state=42)
train_svd = svd.fit_transform(train_tfidf)
train_svd = sk_normalize(train_svd, norm="l2", axis=1).astype(np.float32)
explained = svd.explained_variance_ratio_.sum()
print(f"TruncatedSVD({SVD_DIM}) رویِ train fit شد: شکلِ خروجی {train_svd.shape}, "
      f"واریانسِ توضیح‌داده‌شده={explained:.3f}  ({time.time()-t0:.1f}s)")


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("دستگاه:", DEVICE)

SEED = 42
import random
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

class SparseToDenseMLP(nn.Module):
    def __init__(self, in_dim=300, hidden_dim=768, out_dim=1536, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x):
        return self.net(x)

mlp = SparseToDenseMLP().to(DEVICE)
n_params = sum(p.numel() for p in mlp.parameters())
print(f"✅ MLP ساخته شد: {n_params:,} پارامتر (ByT5 کاملاً فریز — هیچ گرادیانی به آن نمی‌رود)")

def supcon_loss(z, labels, tau=0.07, eps=1e-8):
    B = z.shape[0]
    device = z.device
    z = F.normalize(z, dim=1)
    sim = torch.matmul(z, z.t()) / tau
    labels_eq = labels.unsqueeze(0) == labels.unsqueeze(1)
    self_mask = torch.eye(B, dtype=torch.bool, device=device)
    pos_mask = labels_eq & ~self_mask
    neg_mask = ~labels_eq
    exp_sim = torch.exp(sim)
    neg_sum = (exp_sim * neg_mask.float()).sum(dim=1)
    log_denom = torch.log(exp_sim + neg_sum.unsqueeze(1) + eps)
    log_prob = sim - log_denom
    pos_count = pos_mask.sum(dim=1)
    loss_per_anchor = -(log_prob * pos_mask.float()).sum(dim=1) / (pos_count.float() + eps)
    valid = pos_count > 0
    if valid.sum() == 0:
        return torch.tensor(0.0, device=device, requires_grad=True)
    return loss_per_anchor[valid].mean()

print("✅ supcon_loss تعریف شد (معادلِ اثبات‌شده‌یِ ld_supcon_loss با alpha=beta=gamma=0).")


In [ ]:
import random
import time as _time
from collections import defaultdict

class PKFullSweepSampler:
    def __init__(self, poet_to_indices, P=128, K=4, seed=None):
        self.poet_to_indices = poet_to_indices
        self.P, self.K = P, K
        self.rng = random.Random(seed)

    def _poet_chunks(self, idxs):
        idxs = idxs[:]
        self.rng.shuffle(idxs)
        chunks = [idxs[i:i + self.K] for i in range(0, len(idxs), self.K)]
        if chunks and len(chunks[-1]) < self.K:
            deficit = self.K - len(chunks[-1])
            chunks[-1] = chunks[-1] + [self.rng.choice(idxs) for _ in range(deficit)]
        return chunks

    def __iter__(self):
        poet_queues = {p: self._poet_chunks(idxs) for p, idxs in self.poet_to_indices.items()}
        poets = list(poet_queues.keys())
        while any(poet_queues.values()):
            nonempty = [p for p in poets if poet_queues[p]]
            self.rng.shuffle(nonempty)
            chosen = nonempty[:self.P]
            batch = []
            for p in chosen:
                batch.extend(poet_queues[p].pop())
            yield batch

poet_to_indices = defaultdict(list)
for i, lbl in enumerate(train_labels):
    poet_to_indices[lbl].append(i)

P_PER_BATCH, K_PER_POET = 128, 4
BATCH_SIZE = P_PER_BATCH * K_PER_POET
N_EPOCHS = 20
LR = 1e-3

train_svd_t = torch.from_numpy(train_svd)
train_labels_t = torch.tensor(train_labels, dtype=torch.long)

optimizer = torch.optim.AdamW(mlp.parameters(), lr=LR, weight_decay=1e-4)
n_batches_per_epoch = sum(1 for _ in PKFullSweepSampler(poet_to_indices, P_PER_BATCH, K_PER_POET, seed=0))
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_batches_per_epoch * N_EPOCHS)
print(f"BATCH_SIZE={BATCH_SIZE} | batchهایِ هر epoch ≈ {n_batches_per_epoch} | N_EPOCHS={N_EPOCHS}")

mlp.train()
t_train_start = _time.time()
epoch_losses = []
for epoch in range(N_EPOCHS):
    sampler = PKFullSweepSampler(poet_to_indices, P_PER_BATCH, K_PER_POET, seed=epoch)
    running_loss, n_steps = 0.0, 0
    for batch_idx in sampler:
        bx = train_svd_t[batch_idx].to(DEVICE)
        by = train_labels_t[batch_idx].to(DEVICE)
        optimizer.zero_grad()
        z = mlp(bx)
        loss = supcon_loss(z, by)
        loss.backward()
        optimizer.step()
        scheduler.step()
        running_loss += loss.item()
        n_steps += 1
    avg_loss = running_loss / max(n_steps, 1)
    epoch_losses.append(avg_loss)
    print(f"epoch {epoch+1:02d}/{N_EPOCHS} | supcon_loss={avg_loss:.4f} | "
          f"زمانِ سپری‌شده={_time.time()-t_train_start:.1f}s")

print(f"\n✅ آموزشِ MLP تمام شد. زمانِ کل: {(_time.time()-t_train_start)/60:.1f} دقیقه.")

mlp_ckpt_path = f"{CKPT_DIR}/mlp_fusion_head.pt"
torch.save({"model_state_dict": mlp.state_dict(), "epoch_losses": epoch_losses}, mlp_ckpt_path)
print(f"✅ چک‌پوینتِ MLP ذخیره شد در {mlp_ckpt_path}")


In [ ]:
from collections import defaultdict

def group_by_k(records, embeddings):
    by_k = defaultdict(lambda: {"records": [], "embeddings": []})
    for r, e in zip(records, embeddings):
        by_k[r["k"]]["records"].append(r)
        by_k[r["k"]]["embeddings"].append(e)
    for k in by_k:
        by_k[k]["embeddings"] = np.stack(by_k[k]["embeddings"])
    return dict(sorted(by_k.items()))

test_closed_by_k = group_by_k(test_closed_records, test_closed_embeddings)
test_open_by_k = group_by_k(test_open_records, test_open_embeddings)
K_VALUES = [1, 2, 4, 6]

@torch.no_grad()
def project_sparse_to_dense(texts, batch_size=4096):
    mlp.eval()
    tfidf = vectorizer.transform(texts)
    svd_feat = svd.transform(tfidf)
    svd_feat = sk_normalize(svd_feat, norm="l2", axis=1).astype(np.float32)
    out = np.zeros((len(texts), 1536), dtype=np.float32)
    for start in range(0, len(texts), batch_size):
        end = min(start + batch_size, len(texts))
        bt = torch.from_numpy(svd_feat[start:end]).to(DEVICE)
        z = mlp(bt).float().cpu().numpy()
        out[start:end] = z
    out = sk_normalize(out, norm="l2", axis=1).astype(np.float32)
    return out

closed_sparse_proj_by_k, open_sparse_proj_by_k = {}, {}
for k in K_VALUES:
    t0 = time.time()
    closed_sparse_proj_by_k[k] = project_sparse_to_dense(
        [r["text"] for r in test_closed_by_k[k]["records"]])
    open_sparse_proj_by_k[k] = project_sparse_to_dense(
        [r["text"] for r in test_open_by_k[k]["records"]])
    print(f"k={k}: W_p z_sparse projected — closed {closed_sparse_proj_by_k[k].shape}, "
          f"open {open_sparse_proj_by_k[k].shape}  ({time.time()-t0:.1f}s)")


In [ ]:
import random

def stratified_dev_split(poets, dev_frac=0.15, seed=42):
    rng = random.Random(seed)
    by_poet = defaultdict(list)
    for i, p in enumerate(poets):
        by_poet[p].append(i)
    dev_idx = []
    for p, idxs in by_poet.items():
        idxs = idxs[:]
        rng.shuffle(idxs)
        n_dev = max(1, round(len(idxs) * dev_frac))
        dev_idx.extend(idxs[:n_dev])
    return sorted(dev_idx)

dev_indices = {}
for k in K_VALUES:
    closed_poets_k = [r["poet"] for r in test_closed_by_k[k]["records"]]
    open_poets_k = [r["poet"] for r in test_open_by_k[k]["records"]]
    dev_indices[("closed", k)] = stratified_dev_split(closed_poets_k)
    dev_indices[("open", k)] = stratified_dev_split(open_poets_k)
    print(f"k={k} | dev: closed={len(dev_indices[('closed', k)])}/{len(closed_poets_k)} "
          f"| open={len(dev_indices[('open', k)])}/{len(open_poets_k)}")

with open(f"{PHASE4_DIR}/step1_rrf_fusion_results.json", "r", encoding="utf-8") as f:
    phase41_results = json.load(f)
print("\n✅ نتایجِ فازِ ۴.۱ (RRF) برایِ مقایسه بازخوانی شد — بهترین μ:", phase41_results["best_mu"])


In [ ]:
import numpy as np

def l2_normalize(mat):
    mat = np.asarray(mat, dtype=np.float32)
    return mat / (np.linalg.norm(mat, axis=1, keepdims=True) + 1e-12)

def sweep_lambda_recall1(dense_q, sparse_q, dense_g, sparse_g, query_poets, query_poem_ids,
                          gallery_poets, gallery_poem_ids, lam_q_list, lam_g_list,
                          exclude_same_poem=False, batch_size=500):
    dq = l2_normalize(dense_q); dg = l2_normalize(dense_g)
    sq = l2_normalize(sparse_q); sg = l2_normalize(sparse_g)
    query_poets = np.asarray(query_poets); gallery_poets = np.asarray(gallery_poets)
    query_poem_ids = np.asarray(query_poem_ids); gallery_poem_ids = np.asarray(gallery_poem_ids)
    n_cand = len(lam_q_list)
    need_g_terms = any(abs(lg) > 1e-12 for lg in lam_g_list)
    gs_cross = np.sum(dg * sg, axis=1) if need_g_terms else np.zeros(dg.shape[0], dtype=np.float32)
    hits = np.zeros(n_cand, dtype=np.int64)
    valid_queries = 0
    n_queries = dq.shape[0]
    for start in range(0, n_queries, batch_size):
        end = min(start + batch_size, n_queries)
        bdq, bsq = dq[start:end], sq[start:end]
        Add_dd = bdq @ dg.T
        Add_sd = bsq @ dg.T
        Add_ds = (bdq @ sg.T) if need_g_terms else None
        Add_ss = (bsq @ sg.T) if need_g_terms else None
        qs_cross = np.sum(bdq * bsq, axis=1)
        same_poem_mask = None
        if exclude_same_poem:
            batch_poem_ids = query_poem_ids[start:end]
            same_poem_mask = batch_poem_ids[:, None] == gallery_poem_ids[None, :]
        batch_poets = query_poets[start:end]
        b = end - start
        if start == 0:
            valid_queries = 0
        for ci in range(n_cand):
            lam_q, lam_g = lam_q_list[ci], lam_g_list[ci]
            raw = Add_dd + lam_q * Add_sd
            if need_g_terms and abs(lam_g) > 1e-12:
                raw = raw + lam_g * Add_ds + (lam_q * lam_g) * Add_ss
            norm_q = np.sqrt(1 + 2 * lam_q * qs_cross + lam_q ** 2)
            norm_g = np.sqrt(1 + 2 * lam_g * gs_cross + lam_g ** 2)
            sims = raw / (norm_q[:, None] * norm_g[None, :])
            if exclude_same_poem:
                sims = np.where(same_poem_mask, -np.inf, sims)
            top1 = np.argmax(sims, axis=1)
            for i in range(b):
                if np.isneginf(sims[i, top1[i]]):
                    continue
                if ci == 0:
                    valid_queries += 1
                if gallery_poets[top1[i]] == batch_poets[i]:
                    hits[ci] += 1
    return hits / max(valid_queries, 1), valid_queries

LAMBDA0_CANDIDATES = [0.3, 0.5, 0.75, 1.0, 1.5, 2.0]
DELTA_CANDIDATES = [0.0, 0.1, 0.2, 0.3, 0.5, 0.8]
grid = [(l0, d) for l0 in LAMBDA0_CANDIDATES for d in DELTA_CANDIDATES]
print(f"تعدادِ ترکیب‌هایِ گرید: {len(grid)}")

per_combo_scenario_recall = {g: {} for g in grid}
for k in K_VALUES:
    lam_per_combo = [l0 * np.exp(-d * k) for (l0, d) in grid]
    dev_idx = dev_indices[("closed", k)]
    d = test_closed_by_k[k]
    dense_q = d["embeddings"][dev_idx]
    sparse_q = closed_sparse_proj_by_k[k][dev_idx]
    qp = [d["records"][i]["poet"] for i in dev_idx]
    qpid = [d["records"][i]["poem_id"] for i in dev_idx]
    recalls, _ = sweep_lambda_recall1(
        dense_q, sparse_q, train_embeddings, train_embeddings,
        qp, qpid, train_poets, train_poem_ids,
        lam_q_list=lam_per_combo, lam_g_list=[0.0] * len(grid), exclude_same_poem=False)
    for g, r in zip(grid, recalls):
        per_combo_scenario_recall[g][("closed", k)] = r
    dev_idx = dev_indices[("open", k)]
    d = test_open_by_k[k]
    dense_q = d["embeddings"][dev_idx]
    sparse_q = open_sparse_proj_by_k[k][dev_idx]
    qp = [d["records"][i]["poet"] for i in dev_idx]
    qpid = [d["records"][i]["poem_id"] for i in dev_idx]
    recalls, _ = sweep_lambda_recall1(
        dense_q, sparse_q, dense_q, sparse_q,
        qp, qpid, qp, qpid,
        lam_q_list=lam_per_combo, lam_g_list=lam_per_combo, exclude_same_poem=True)
    for g, r in zip(grid, recalls):
        per_combo_scenario_recall[g][("open", k)] = r
    print(f"✅ جاروبِ (λ0,δ) برایِ k={k} (closed+open، هر {len(grid)} ترکیب) تمام شد.")

best_combo, best_avg = None, -1
for g in grid:
    avg = float(np.mean(list(per_combo_scenario_recall[g].values())))
    if avg > best_avg:
        best_avg, best_combo = avg, g
best_lambda0, best_delta = best_combo
print(f"\n🏆 بهترینِ سراسری: λ0={best_lambda0}, δ={best_delta} "
      f"(میانگینِ Recall@1 رویِ dev = {best_avg:.4f})")

print("\n-- چندتایِ برتر برایِ شفافیت --")
ranked = sorted(grid, key=lambda g: -float(np.mean(list(per_combo_scenario_recall[g].values()))))
for g in ranked[:5]:
    avg = float(np.mean(list(per_combo_scenario_recall[g].values())))
    print(f"  λ0={g[0]}, δ={g[1]} → میانگینِ dev Recall@1={avg:.4f}")


In [ ]:
import scipy.sparse as sp

def compute_sims(batch_q, g):
    sims = batch_q @ g.T
    return np.asarray(sims, dtype=np.float32)

def evaluate_retrieval(query_vecs, query_poets, query_poem_ids,
                        gallery_vecs, gallery_poets, gallery_poem_ids,
                        ks=(1, 5, 10), exclude_same_poem=False, batch_size=500):
    q = l2_normalize(query_vecs); g = l2_normalize(gallery_vecs)
    query_poets = np.asarray(query_poets); gallery_poets = np.asarray(gallery_poets)
    query_poem_ids = np.asarray(query_poem_ids); gallery_poem_ids = np.asarray(gallery_poem_ids)
    max_k = max(ks); n_gallery = g.shape[0]
    recall_hits = {k: 0 for k in ks}; mrr_total = 0.0; valid_queries = 0
    n_queries = q.shape[0]
    for start in range(0, n_queries, batch_size):
        end = min(start + batch_size, n_queries)
        batch_q = q[start:end]
        sims = compute_sims(batch_q, g)
        if exclude_same_poem:
            batch_poem_ids = query_poem_ids[start:end]
            same_poem_mask = batch_poem_ids[:, None] == gallery_poem_ids[None, :]
            sims = np.where(same_poem_mask, -np.inf, sims)
        k_eff = min(max_k, n_gallery - 1) if exclude_same_poem else min(max_k, n_gallery)
        top_idx = np.argpartition(-sims, kth=k_eff - 1, axis=1)[:, :k_eff]
        row_idx = np.arange(top_idx.shape[0])[:, None]
        top_sims = sims[row_idx, top_idx]
        order = np.argsort(-top_sims, axis=1)
        top_idx_sorted = top_idx[row_idx, order]
        top_sims_sorted = top_sims[row_idx, order]
        batch_poets = query_poets[start:end]
        for i in range(end - start):
            if np.all(np.isneginf(top_sims_sorted[i])): continue
            valid_queries += 1
            retrieved_poets = gallery_poets[top_idx_sorted[i]]
            hit_positions = np.where(retrieved_poets == batch_poets[i])[0]
            if len(hit_positions) > 0:
                rank = hit_positions[0] + 1
                mrr_total += 1.0 / rank
                for k in ks:
                    if rank <= k: recall_hits[k] += 1
    out = {"n_queries": valid_queries, "mrr": mrr_total / valid_queries if valid_queries else float("nan")}
    for k in ks: out[f"recall@{k}"] = recall_hits[k] / valid_queries if valid_queries else float("nan")
    return out

mlp_results = {}
for k in K_VALUES:
    lam = best_lambda0 * np.exp(-best_delta * k)
    _t0 = time.time()
    d = test_closed_by_k[k]
    hybrid_q = l2_normalize(l2_normalize(d["embeddings"]) + lam * closed_sparse_proj_by_k[k])
    qp = [r["poet"] for r in d["records"]]; qpid = [r["poem_id"] for r in d["records"]]
    mlp_results[("closed", k)] = evaluate_retrieval(
        hybrid_q, qp, qpid, train_embeddings, train_poets, train_poem_ids, exclude_same_poem=False)
    print(f"  [k={k}] closed-MLPHybrid  در {time.time()-_t0:.1f}s تمام شد")
    _t0 = time.time()
    d = test_open_by_k[k]
    hybrid_qg = l2_normalize(l2_normalize(d["embeddings"]) + lam * open_sparse_proj_by_k[k])
    qp = [r["poet"] for r in d["records"]]; qpid = [r["poem_id"] for r in d["records"]]
    mlp_results[("open", k)] = evaluate_retrieval(
        hybrid_qg, qp, qpid, hybrid_qg, qp, qpid, exclude_same_poem=True)
    print(f"  [k={k}] open-MLPHybrid    در {time.time()-_t0:.1f}s تمام شد")
    print(f"k={k} | closed R@1={mlp_results[('closed',k)]['recall@1']:.4f} "
          f"| open R@1={mlp_results[('open',k)]['recall@1']:.4f}  (λ={lam:.4f})")

print("\n=== جدولِ مقایسه‌یِ نهایی (Recall@1) — PACL / TF-IDF / RRF (فازِ ۴.۱) / MLP-Hybrid (فازِ ۴.۲) ===")
header = f"{'scenario':<8}{'k':<4}{'PACL':<9}{'TF-IDF':<9}{'RRF':<9}{'MLP-Hybrid':<12}{'بهبود نسبت به بهترینِ قبلی'}"
print(header)
final_rows = []
for scenario in ["closed", "open"]:
    for k in K_VALUES:
        pacl = phase41_results["baseline_results"][f"{scenario}_{k}_pacl"]["recall@1"]
        tfidf = phase41_results["baseline_results"][f"{scenario}_{k}_tfidf"]["recall@1"]
        rrf = phase41_results["rrf_results"][f"{scenario}_{k}"]["recall@1"]
        mlp_r1 = mlp_results[(scenario, k)]["recall@1"]
        best_prev = max(pacl, tfidf, rrf)
        delta = mlp_r1 - best_prev
        final_rows.append((scenario, k, pacl, tfidf, rrf, mlp_r1, delta))
        print(f"{scenario:<8}{k:<4}{pacl:<9.4f}{tfidf:<9.4f}{rrf:<9.4f}{mlp_r1:<12.4f}{delta:+.4f}")

mlp_open_k1 = mlp_results[("open", 1)]["recall@1"]
mlp_open_k6 = mlp_results[("open", 6)]["recall@1"]
print(f"\n🎯 هدفِ  — Open k=1: Recall@1 ≥ 0.385  | نتیجه: {mlp_open_k1:.4f}  "
      f"{'✅ محقق شد' if mlp_open_k1 >= 0.385 else '❌ محقق نشد'}")
print(f"🎯 هدفِ — Open k=6: Recall@1 ≥ 0.635  | نتیجه: {mlp_open_k6:.4f}  "
      f"{'✅ محقق شد' if mlp_open_k6 >= 0.635 else '❌ محقق نشد'}")


In [ ]:
import json as _json

def pack(res_dict):
    return {"_".join(str(x) for x in key): res for key, res in res_dict.items()}

summary = {
    "method": "MLP Fusion Head — z_hybrid = Normalize(z_dense + lambda(k) * MLP(SVD300(TF-IDF)))",
    "formula": "lambda(k) = lambda0 * exp(-delta*k); gallery lambda=0 for Closed (train=full poems)",
    "architecture": {"svd_dim": 300, "mlp_hidden_dim": 768, "mlp_out_dim": 1536,
                      "loss": "vanilla SupCon (alpha=beta=gamma=0, equivalent to ld_supcon_loss)",
                      "batch_size": BATCH_SIZE, "p_per_batch": P_PER_BATCH, "k_per_poet": K_PER_POET,
                      "n_epochs": N_EPOCHS, "lr": LR, "final_train_loss": epoch_losses[-1]},
    "lambda_sweep": {"lambda0_candidates": LAMBDA0_CANDIDATES, "delta_candidates": DELTA_CANDIDATES,
                      "best_lambda0": best_lambda0, "best_delta": best_delta,
                      "best_dev_avg_recall1": best_avg},
    "mlp_results": pack(mlp_results),
    "comparison_vs_phase41": [
        {"scenario": s, "k": k, "pacl": p, "tfidf": t, "rrf": r, "mlp_hybrid": m, "improvement_vs_best_prev": d}
        for (s, k, p, t, r, m, d) in final_rows
    ],
    "professor_targets": {
        "open_k1_recall1_target": 0.385,
        "open_k1_recall1_actual": mlp_open_k1,
        "open_k6_recall1_target": 0.635,
        "open_k6_recall1_actual": mlp_open_k6,
    },
}

out_path = f"{PHASE4_DIR}/step2_mlp_fusion_results.json"
with open(out_path, "w", encoding="utf-8") as f:
    _json.dump(summary, f, ensure_ascii=False, indent=2)

print(f"✅ ذخیره شد در {out_path}")
